In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
================================================================================
 Process a P-Dependent 5-Fold CV results workbook with analysis sheets.
================================================================================

Takes the filepath of an existing results workbook (as produced by
P_dep_TRAIN_GenearteReport / evaluate_pdep_train_5fold.py -- a "Results" sheet
with columns Fold | System | Rule | p | EER (%) | Rank-1 @ FAR=1% | N_train)
and ADDS the following sheets, computed entirely from that data. Any sheet
that already exists (e.g. from a previous run of this script) is replaced
rather than duplicated; every other sheet in the workbook is left untouched.

  1. Methodology                  - detailed description of data and methods
  2. Best_By_EER                  - per-system winner selected on mean EER
  3. Best_by_R1FAR1                - per-system winner selected on mean Rank-1@FAR1%
  4. EER_Sensitivity_Analysis      - mean EER per System x Rule (averaged over p)
  5. R1FAR1_Sensitivity_Analysis   - mean Rank-1@FAR1% per System x Rule (averaged over p)
  6. P_Sensitivity_Analysis        - EER-optimal p per Rule x System
  7. BestEER_vs_P_AllRules         - best achievable EER vs p, min over all 10 rules
  8. EER_vs_P_WinningRule          - EER vs p, using each system's own winning rule

USAGE
-----
    python augment_pdep_report.py PDependent_5FoldCV_seed62.xlsx

    # or import and call directly:
    from augment_pdep_report import augment_workbook
    augment_workbook("PDependent_5FoldCV_seed42.xlsx")
================================================================================
"""

from __future__ import annotations

import sys
from pathlib import Path
from typing import List

import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.formatting.rule import ColorScaleRule
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.worksheet import Worksheet


# ==============================================================================
#  CONFIGURATION
# ==============================================================================

RESULTS_XLSX_FILE = "./Results/PDependent_5FoldCV_seed42.xlsx"
RESULTS_SHEET_NAME = "Results"       # sheet holding the raw Fold/System/Rule/p data
RESULTS_HEADER_ROW = 3               # 0-indexed: title, subtitle, blank, HEADER, data...
FAR_LABEL = "Rank-1 @ FAR=1%"        # column name as it appears in the source sheet
EER_LABEL = "EER (%)"

THEME_COLOR = "4A148C"               # distinct purple for the analysis sheets
FONT_NAME = "Arial"


# ==============================================================================
#  STEP 0 -- LOAD THE SOURCE DATA
# ==============================================================================

def load_results(filepath: Path) -> pd.DataFrame:
    """
    Read the raw (Fold, System, Rule, p, EER, Rank-1) table from the workbook's
    Results sheet, using the header-row convention this family of scripts uses
    (two title rows, a blank row, then the header).
    """
    frame = pd.read_excel(filepath, sheet_name=RESULTS_SHEET_NAME,
                          header=RESULTS_HEADER_ROW)
    required = {"Fold", "System", "Rule", "p", EER_LABEL, FAR_LABEL, "N_train"}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(
            f"Results sheet is missing expected columns: {sorted(missing)}. "
            f"Found columns: {list(frame.columns)}"
        )
    return frame


def system_order(frame: pd.DataFrame) -> List[str]:
    """Canonical S1..S8 order, restricted to systems actually present."""
    canonical = [f"S{i}" for i in range(1, 9)]
    present = set(frame["System"])
    return [s for s in canonical if s in present]


def rule_order(frame: pd.DataFrame) -> List[str]:
    """Canonical rule order (as first encountered in the data), stable across runs."""
    canonical = ["Zimmermann1", "Zimmermann2", "Luhandjula", "Werners1", "Werners2",
                "Sales1", "Sales2", "Mizumoto1", "Mizumoto2", "R5p"]
    present = set(frame["Rule"])
    ordered = [r for r in canonical if r in present]
    # Include any rule not in the canonical list too, so nothing is silently dropped.
    ordered += sorted(present - set(ordered))
    return ordered


# ==============================================================================
#  STEP 1 -- AGGREGATE ACROSS FOLDS (mean-first, as the corrected protocol requires)
# ==============================================================================

def aggregate_per_config(frame: pd.DataFrame) -> pd.DataFrame:
    """
    Average EER and Rank-1 across the 5 folds for every (System, Rule, p).

    This is the base table every summary below is built from. Averaging BEFORE
    selecting a winner (rather than selecting per fold and averaging after) is
    what keeps every reported winner a single, fixed, deployable configuration.
    """
    grouped = frame.groupby(["System", "Rule", "p"], as_index=False).agg(
        EER_mean=(EER_LABEL, "mean"),
        EER_std=(EER_LABEL, "std"),
        Rank1_mean=(FAR_LABEL, "mean"),
        Rank1_std=(FAR_LABEL, "std"),
        N_folds=("Fold", "count"),
    )
    grouped["EER_std"] = grouped["EER_std"].fillna(0.0)
    grouped["Rank1_std"] = grouped["Rank1_std"].fillna(0.0)
    return grouped


# ==============================================================================
#  STEP 2 -- BUILD EACH REQUESTED TABLE
# ==============================================================================

def build_best_by_eer(per_config: pd.DataFrame, systems: List[str]) -> pd.DataFrame:
    """
    Sheet 2: for each system, the single (rule, p) with the lowest mean EER
    (aggregated across folds). Reports EER and Rank-1 at THAT configuration.
    """
    rows = []
    for system in systems:
        subset = per_config[per_config.System == system]
        best = subset.loc[subset.EER_mean.idxmin()]
        rows.append({
            "System": system,
            "Rule": best.Rule,
            "p": best.p,
            "EER (%)": round(best.EER_mean, 4),
            "EER_std": round(best.EER_std, 4),
#            "Rank-1 (%)": round(best.Rank1_mean, 4),
            "Rank-1 @ FAR=1%": round(best.Rank1_mean, 4),
            "R1FAR1_std": round(best.Rank1_std, 4),
        })
    return pd.DataFrame(rows)


def build_best_by_rank1(per_config: pd.DataFrame, systems: List[str]) -> pd.DataFrame:
    """
    Sheet 3: for each system, the single (rule, p) with the highest mean
    Rank-1 @ FAR=1% (aggregated across folds) -- an INDEPENDENT optimization
    from Best_By_EER, so the winning (rule, p) can differ between the two.
    """
    rows = []
    for system in systems:
        subset = per_config[per_config.System == system]
        best = subset.loc[subset.Rank1_mean.idxmax()]
        rows.append({
            "System": system,
            "Rule": best.Rule,
            "p": best.p,
            "EER (%)": round(best.EER_mean, 4),
            "EER_std": round(best.EER_std, 4),
            "Rank-1 (%)": round(best.Rank1_mean, 4),
            "Rank-1 @ FAR=1%": round(best.Rank1_mean, 4),
            "R1FAR1_std": round(best.Rank1_std, 4),
        })
    return pd.DataFrame(rows)


def build_eer_sensitivity(per_config: pd.DataFrame, systems: List[str],
                          rules: List[str]) -> pd.DataFrame:
    """
    Sheet 4: mean EER per (System, Rule), averaged over EVERY value of p (and
    over folds). This shows how a rule performs "on average" for a system,
    independent of how well p happens to be tuned -- a measure of how
    sensitive the system is to the choice of rule itself.
    """
    pivot = (
        per_config.groupby(["System", "Rule"])["EER_mean"].min()
        .unstack("Rule").reindex(index=systems, columns=rules)
    )
    pivot.insert(0, "System", pivot.index)
    return pivot.reset_index(drop=True)


def build_rank1_sensitivity(per_config: pd.DataFrame, systems: List[str],
                            rules: List[str]) -> pd.DataFrame:
    """Sheet 5: same idea as Sheet 4, for mean Rank-1 @ FAR=1% instead of EER."""
    pivot = (
        per_config.groupby(["System", "Rule"])["Rank1_mean"].max()
        .unstack("Rule").reindex(index=systems, columns=rules)
    )
    pivot.insert(0, "System", pivot.index)
    return pivot.reset_index(drop=True)


def build_p_sensitivity(per_config: pd.DataFrame, systems: List[str],
                        rules: List[str]) -> pd.DataFrame:
    """
    Sheet 6: for every (Rule, System), the p that minimizes mean EER.
    Rows = rules, columns = systems -- lets you scan whether a rule's optimal
    p is stable across systems or jumps around.
    """
    idx = per_config.groupby(["System", "Rule"])["EER_mean"].idxmin()
    best_p = per_config.loc[idx, ["System", "Rule", "p"]]
    pivot = best_p.pivot(index="Rule", columns="System", values="p").reindex(
        index=rules, columns=systems
    )
    pivot.insert(0, "Rule", pivot.index)
    return pivot.reset_index(drop=True)


def build_best_eer_vs_p(per_config: pd.DataFrame, systems: List[str],
                        p_values: List[float]) -> pd.DataFrame:
    """
    Sheet 7: for every p, the BEST achievable mean EER at that p -- minimized
    over all 10 rules, independently for each system. Answers "if I could pick
    the best rule for this exact p, how good could this system get?".
    """
    pivot = (
        per_config.groupby(["System", "p"])["EER_mean"].min()
        .unstack("System").reindex(index=p_values, columns=systems)
    )
    pivot.insert(0, "p", pivot.index)
    return pivot.reset_index(drop=True)


def build_eer_vs_p_winning_rule(per_config: pd.DataFrame, best_by_eer: pd.DataFrame,
                                systems: List[str], p_values: List[float]) -> pd.DataFrame:
    """
    Sheet 8: for every p, the mean EER obtained by FIXING each system's own
    winning rule (from Best_By_EER) and only varying p. Unlike Sheet 7, the
    rule does not change -- this shows how forgiving (or not) a system's
    chosen rule is to a suboptimal p.
    """
    winning_rule = dict(zip(best_by_eer.System, best_by_eer.Rule))
    columns = {}
    for system in systems:
        rule = winning_rule[system]
        subset = per_config[(per_config.System == system) & (per_config.Rule == rule)]
        series = subset.set_index("p")["EER_mean"].reindex(p_values)
        columns[system] = series
    result = pd.DataFrame(columns)
    result.insert(0, "p", result.index)
    return result.reset_index(drop=True)


# ==============================================================================
#  STEP 3 -- METHODOLOGY TEXT
# ==============================================================================

def methodology_lines(frame: pd.DataFrame, systems: List[str], rules: List[str],
                      p_values: List[float]) -> List[tuple]:
    """Return (kind, text) pairs describing the analysis, for the README-style sheet."""
    n_folds = int(frame["Fold"].nunique())
    p_min, p_max = min(p_values), max(p_values)
    p_step = round(p_values[1] - p_values[0], 10) if len(p_values) > 1 else 0.0

    return [
        ("title", "Methodology"),
        ("note",  "How every derived sheet in this workbook was computed from the Results sheet."),
        ("blank", ""),
        ("head",  "SOURCE DATA"),
        ("body",  f"The 'Results' sheet holds {len(frame)} raw rows: {n_folds} cross-validation "
                  f"folds x {len(systems)} fusion systems x {len(rules)} parametrized rules x "
                  f"{len(p_values)} values of p (p = {p_min:.2f} to {p_max:.2f}, step {p_step:.2f}). "
                  "Each row is the EER and Rank-1 @ FAR=1% obtained by fusing two (or, for cascaded "
                  "systems, three) score modalities with that (rule, p), measured on ONE fold's "
                  "80% TRAIN subset."),
        ("body",  "Folds are disjoint 20% subject partitions; each fold's 80% complement is its "
                  "TRAIN set. Min-max normalization is fit independently on each fold's TRAIN "
                  "subjects and applied to that fold only -- no fold ever influences another "
                  "fold's scale."),
        ("blank", ""),
        ("head",  "AGGREGATION ORDER (why this matters)"),
        ("body",  "For every summary below, results are averaged ACROSS THE 5 FOLDS FIRST, for "
                  "each (System, Rule, p) independently. Only after that averaging does any "
                  "'best' selection happen -- picking the best p for a rule, or the best rule for "
                  "a system, is always done on the mean across folds."),
        ("body",  "This ordering is deliberate: selecting a different p inside each fold and "
                  "averaging the results afterwards would blend several different configurations "
                  "into one number, which is optimistically biased and corresponds to no single "
                  "system you could actually deploy. Every winner reported here is one fixed "
                  "(rule, p) pair, and its reported EER/Rank-1 is the honest mean of that SAME "
                  "configuration evaluated independently in all 5 folds."),
        ("blank", ""),
        ("head",  "SHEET-BY-SHEET DEFINITIONS"),
        ("body",  "Best_By_EER: per system, the (rule, p) with the lowest mean EER across folds. "
                  "EER and Rank-1 reported are both measured at that same, single configuration."),
        ("body",  "Best_by_R1FAR1: per system, the (rule, p) with the highest mean Rank-1 @ FAR=1% "
                  "across folds -- an INDEPENDENT optimization from Best_By_EER. The winning "
                  "configuration, and therefore the reported EER, can differ between the two sheets: "
                  "the p that minimizes EER is not always the p that maximizes Rank-1 at a fixed FAR."),
        ("body",  "EER_Sensitivity_Analysis / R1FAR1_Sensitivity_Analysis: mean EER / Rank-1 for "
                  "every (System, Rule), averaged over ALL values of p (not just the optimal one). "
                  "This shows how a rule performs 'on average' for a system, isolating the effect "
                  "of the rule choice from the effect of tuning p well."),
        ("body",  "P_Sensitivity_Analysis: the EER-optimal p for every (Rule, System) pair, laid "
                  "out as Rule x System. Reading across a row shows whether a rule's best p is "
                  "stable across systems (a robust, transferable setting) or varies a lot (a sign "
                  "the optimal p is data-dependent and less trustworthy)."),
        ("body",  "BestEER_vs_P_AllRules: for each p, the best mean EER ACHIEVABLE BY ANY RULE at "
                  "that p, per system -- i.e. 'if I could always pick the best rule for this p, how "
                  "good could I get?'. Shows the ceiling performance as a function of p alone."),
        ("body",  "EER_vs_P_WinningRule: for each p, the mean EER obtained using each system's OWN "
                  "winning rule (from Best_By_EER) held FIXED while p varies. Unlike the previous "
                  "sheet, the rule never changes here -- this shows how forgiving that specific, "
                  "already-chosen rule is to a suboptimal p, which is the more realistic question "
                  "once a rule has actually been selected for deployment."),
        ("blank", ""),
        ("head",  "ON THE 'Rank-1 (%)' AND 'Rank-1 @ FAR=1%' COLUMNS"),
        ("body",  "The source Results sheet contains a single identification metric: Rank-1 "
                  "measured at a 1% False Accept Rate (open-set identification -- a probe counts "
                  "as correct only if its top match is right AND clears a threshold set so 1% of "
                  "impostor comparisons would be accepted). It does NOT contain a threshold-free "
                  "closed-set Rank-1 rate, which would require re-running against the original "
                  "score matrices. The 'Rank-1 (%)' and 'Rank-1 @ FAR=1%' columns in the summary "
                  "sheets below therefore report the SAME value; this is stated here explicitly "
                  "rather than silently duplicating a number under two different-looking labels."),
    ]


# ==============================================================================
#  STEP 4 -- EXCEL WRITING HELPERS
# ==============================================================================

def _styles():
    thin = Side(style="thin", color="D9D9D9")
    return {
        "header_fill": PatternFill("solid", fgColor=THEME_COLOR),
        "header_font": Font(name=FONT_NAME, bold=True, color="FFFFFF", size=10),
        "title_font":  Font(name=FONT_NAME, bold=True, size=13, color=THEME_COLOR),
        "note_font":   Font(name=FONT_NAME, italic=True, size=9, color="555555"),
        "cell_font":   Font(name=FONT_NAME, size=10),
        "border":      Border(left=thin, right=thin, top=thin, bottom=thin),
    }


def _autofit(worksheet: Worksheet, n_columns: int, header_row: int, n_rows: int,
            minimum: int = 9, maximum: int = 32) -> None:
    for col_idx in range(1, n_columns + 1):
        letter = get_column_letter(col_idx)
        longest = max(
            (len(str(worksheet.cell(row=r, column=col_idx).value))
             for r in range(header_row, header_row + n_rows + 1)
             if worksheet.cell(row=r, column=col_idx).value is not None),
            default=minimum,
        )
        worksheet.column_dimensions[letter].width = min(max(longest + 2, minimum), maximum)


def _write_table(worksheet: Worksheet, frame: pd.DataFrame, styles: dict,
                 title: str, subtitle: str = "", percent_cols: tuple = ()) -> int:
    worksheet.cell(row=1, column=1, value=title).font = styles["title_font"]
    row = 2
    if subtitle:
        worksheet.cell(row=row, column=1, value=subtitle).font = styles["note_font"]
        row += 1
    row += 1
    header_row = row

    for col_idx, column in enumerate(frame.columns, start=1):
        cell = worksheet.cell(row=header_row, column=col_idx, value=str(column))
        cell.fill = styles["header_fill"]
        cell.font = styles["header_font"]
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border = styles["border"]

    for offset, record in enumerate(frame.itertuples(index=False), start=1):
        for col_idx, value in enumerate(record, start=1):
            if isinstance(value, (np.integer,)):
                value = int(value)
            elif isinstance(value, (np.floating,)):
                value = None if np.isnan(value) else float(value)
            cell = worksheet.cell(row=header_row + offset, column=col_idx, value=value)
            cell.font = styles["cell_font"]
            cell.border = styles["border"]
            column = frame.columns[col_idx - 1]
            if column in percent_cols:
                cell.number_format = "0.0000"
            elif column == "p":
                cell.number_format = "0.00"

    worksheet.freeze_panes = worksheet.cell(row=header_row + 1, column=1)
    _autofit(worksheet, len(frame.columns), header_row, len(frame))
    return header_row


def _write_methodology(worksheet: Worksheet, lines: List[tuple], styles: dict) -> None:
    worksheet.column_dimensions["A"].width = 108
    fonts = {
        "title": styles["title_font"],
        "head":  Font(name=FONT_NAME, bold=True, size=11),
        "note":  styles["note_font"],
        "body":  styles["cell_font"],
        "blank": styles["cell_font"],
    }
    for row, (kind, text) in enumerate(lines, start=1):
        cell = worksheet.cell(row=row, column=1, value=text)
        cell.font = fonts[kind]
        cell.alignment = Alignment(wrap_text=False, vertical="top")


def _replace_sheet(workbook, name: str):
    """Remove a sheet if present, then create and return a fresh one with that name."""
    if name in workbook.sheetnames:
        del workbook[name]
    return workbook.create_sheet(name[:31])


# ==============================================================================
#  MAIN ENTRY POINT
# ==============================================================================

def augment_workbook(filepath: str) -> Path:
    """
    Add (or refresh) the eight analysis sheets in the workbook at `filepath`,
    computed from its existing Results sheet. Returns the path written to.
    """
    path = Path(filepath)
    if not path.exists():
        raise FileNotFoundError(f"Workbook not found: {path.resolve()}")

    frame = load_results(path)
    systems = system_order(frame)
    rules = rule_order(frame)
    p_values = sorted(frame["p"].unique().tolist())

    per_config = aggregate_per_config(frame)

    best_by_eer = build_best_by_eer(per_config, systems)
    best_by_rank1 = build_best_by_rank1(per_config, systems)
    eer_sensitivity = build_eer_sensitivity(per_config, systems, rules)
    rank1_sensitivity = build_rank1_sensitivity(per_config, systems, rules)
    p_sensitivity = build_p_sensitivity(per_config, systems, rules)
    best_eer_vs_p = build_best_eer_vs_p(per_config, systems, p_values)
    eer_vs_p_winning = build_eer_vs_p_winning_rule(per_config, best_by_eer, systems, p_values)

    workbook = load_workbook(path)
    styles = _styles()

    # 1. Methodology
    ws = _replace_sheet(workbook, "Methodology")
    _write_methodology(ws, methodology_lines(frame, systems, rules, p_values), styles)

    # 2. Best_By_EER
    ws = _replace_sheet(workbook, "Best_By_EER")
    _write_table(
        ws, best_by_eer, styles,
        title="Best Configuration per System \u2014 selected on minimum mean EER",
        subtitle="Rule and p chosen on the mean across all 5 folds; EER and Rank-1 reported at that same configuration.",
        percent_cols=("EER (%)", "Rank-1 (%)", "Rank-1 @ FAR=1%"),
    )

    # 3. Best_by_R1FAR1
    ws = _replace_sheet(workbook, "Best_by_R1FAR1")
    _write_table(
        ws, best_by_rank1, styles,
        title="Best Configuration per System \u2014 selected on maximum mean Rank-1 @ FAR=1%",
        subtitle="Independent optimization from Best_By_EER; the winning (rule, p) can differ.",
        percent_cols=("EER (%)", "Rank-1 (%)", "Rank-1 @ FAR=1%"),
    )

    # 4. EER_Sensitivity_Analysis
    ws = _replace_sheet(workbook, "EER_Sensitivity_Analysis")
    hr = _write_table(
        ws, eer_sensitivity, styles,
        title="EER Sensitivity Analysis \u2014 mean EER (%) per System x Rule, averaged over all p",
        subtitle="Shows how a rule performs on average for a system, independent of how well p is tuned.",
        percent_cols=tuple(rules),
    )
    if len(rules) > 1:
        last_col = get_column_letter(len(eer_sensitivity.columns))
        ws.conditional_formatting.add(
            f"B{hr+1}:{last_col}{hr+len(eer_sensitivity)}",
            ColorScaleRule(start_type="min", start_color="63BE7B",
                          end_type="max", end_color="F8696B"),
        )

    # 5. R1FAR1_Sensitivity_Analysis
    ws = _replace_sheet(workbook, "R1FAR1_Sensitivity_Analysis")
    hr = _write_table(
        ws, rank1_sensitivity, styles,
        title="Rank-1 @ FAR=1% Sensitivity Analysis \u2014 mean per System x Rule, averaged over all p",
        subtitle="Shows how a rule performs on average for a system, independent of how well p is tuned.",
        percent_cols=tuple(rules),
    )
    if len(rules) > 1:
        last_col = get_column_letter(len(rank1_sensitivity.columns))
        ws.conditional_formatting.add(
            f"B{hr+1}:{last_col}{hr+len(rank1_sensitivity)}",
            ColorScaleRule(start_type="min", start_color="F8696B",
                          end_type="max", end_color="63BE7B"),
        )

    # 6. P_Sensitivity_Analysis
    ws = _replace_sheet(workbook, "P_Sensitivity_Analysis")
    hr = _write_table(
        ws, p_sensitivity, styles,
        title="P-Sensitivity Analysis \u2014 EER-optimal p per Rule x System",
        subtitle="A stable value across a row means that rule's optimal p transfers well across systems.",
        percent_cols=(),
    )
    # p values (not percentages) live in the system columns here.
    for row in ws.iter_rows(min_row=hr + 1, min_col=2, max_col=1 + len(systems)):
        for cell in row:
            cell.number_format = "0.00"

    # 7. BestEER_vs_P_AllRules
    ws = _replace_sheet(workbook, "BestEER_vs_P_AllRules")
    _write_table(
        ws, best_eer_vs_p, styles,
        title="Best Achievable EER (%) vs p \u2014 minimum over all 10 rules (5-fold CV mean)",
        subtitle="Per system, the lowest mean EER obtainable at each p if the best rule for that p is used.",
        percent_cols=tuple(systems),
    )

    # 8. EER_vs_P_WinningRule
    ws = _replace_sheet(workbook, "EER_vs_P_WinningRule")
    _write_table(
        ws, eer_vs_p_winning, styles,
        title="EER (%) vs p \u2014 each system's winning rule held fixed (5-fold CV mean)",
        subtitle="Uses ONLY each system's Best_By_EER rule at every p; shows sensitivity to p once the rule is fixed.",
        percent_cols=tuple(systems),
    )

    # Keep the original Results sheet first, analysis sheets after it, in a
    # sensible reading order.
    desired_order = [RESULTS_SHEET_NAME, "Methodology", "Best_By_EER", "Best_by_R1FAR1",
                     "EER_Sensitivity_Analysis", "R1FAR1_Sensitivity_Analysis",
                     "P_Sensitivity_Analysis", "BestEER_vs_P_AllRules",
                     "EER_vs_P_WinningRule"]
    remaining = [s for s in workbook.sheetnames if s not in desired_order]
    workbook._sheets = [workbook[s] for s in desired_order if s in workbook.sheetnames] \
                       + [workbook[s] for s in remaining]

    workbook.save(path)
    return path


if __name__ == "__main__":

    filepath = RESULTS_XLSX_FILE
    print(f"Augmenting workbook: {filepath}")
    output_path = augment_workbook(filepath)
    print("Done. Sheets added/updated: Methodology, Best_By_EER, Best_by_R1FAR1, "
          "EER_Sensitivity_Analysis, R1FAR1_Sensitivity_Analysis, "
          "P_Sensitivity_Analysis, BestEER_vs_P_AllRules, EER_vs_P_WinningRule")
    print(f"Saved to: {output_path}")
    

Augmenting workbook: ./Results/PDependent_5FoldCV_seed42.xlsx
Done. Sheets added/updated: Methodology, Best_By_EER, Best_by_R1FAR1, EER_Sensitivity_Analysis, R1FAR1_Sensitivity_Analysis, P_Sensitivity_Analysis, BestEER_vs_P_AllRules, EER_vs_P_WinningRule
Saved to: Results\PDependent_5FoldCV_seed42.xlsx
